In [ ]:
# AI Job Market Intelligence Platform
## تحليل شامل لسوق العمل في المملكة المتحدة
منصة تحليل بيانات متكاملة تهدف إلى دراسة سوق العمل في المملكة المتحدة باستخدام **بيانات حقيقية من Adzuna API**، واستخراج أهم المهارات والاتجاهات المطلوبة في سوق العمل، مع تطبيق تقنيات **NLP** و **Machine Learning**.
## 📌 Project Sections


In [ ]:
import os
import re
import time
import requests
import pandas as pd
import sqlalchemy as sa
from dotenv import load_dotenv
from collections import Counter

load_dotenv("../.env")
APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")
SERVER_NAME = "localhost\\SQLEXPRESS"

In [4]:
### 1. Data Collection
##جمع بيانات الوظائف من **Adzuna API** وتنظيمها في DataFrame.

def get_jobs_for_keyword(keyword, max_pages=3, max_retries=3):
    all_jobs = []
    for page in range(1, max_pages + 1):
        url = f"https://api.adzuna.com/v1/api/jobs/gb/search/{page}"
        params = {
            "app_id": APP_ID, "app_key": APP_KEY,
            "results_per_page": 20, "what": keyword,
            "content-type": "application/json"
        }
        success = False
        for attempt in range(1, max_retries + 1):
            response = requests.get(url, params=params)
            if response.status_code == 200:
                success = True
                break
            time.sleep(5)
        if not success:
            break
        results = response.json().get("results", [])
        if not results:
            break
        all_jobs.extend(results)
        time.sleep(3)
    return all_jobs

In [5]:
##Collect All Job Families
keywords = [
    "data analyst", "data scientist", "data engineer",
    "machine learning engineer", "business analyst",
    "business intelligence analyst", "software engineer",
    "cloud engineer", "cybersecurity engineer", "AI engineer",
]

all_jobs_raw = []
for kw in keywords:
    jobs = get_jobs_for_keyword(kw, max_pages=7)
    for j in jobs:
        j["search_keyword"] = kw
    all_jobs_raw.extend(jobs)

print("Total collected:", len(all_jobs_raw))

Total collected: 1388


In [7]:
##Clean & Deduplicate
rows = [{
    "job_id": j.get("id"), "title": j.get("title"),
    "company": j.get("company", {}).get("display_name"),
    "location": j.get("location", {}).get("display_name"),
    "salary_min": j.get("salary_min"), "salary_max": j.get("salary_max"),
    "contract_type": j.get("contract_type"),
    "category": j.get("category", {}).get("label"),
    "description": j.get("description"),
    "search_keyword": j.get("search_keyword"), "created": j.get("created"),
} for j in all_jobs_raw]

df_clean = pd.DataFrame(rows).drop_duplicates(subset=["job_id"], keep="first")
df_clean.to_csv("../data/processed/jobs_master.csv", index=False, encoding="utf-8-sig")
print("Unique jobs:", df_clean.shape[0])

Unique jobs: 1353


In [8]:
##Skills Extraction (NLP)
skills_list = [
    "Python", "SQL", "Java", "Scala", "Excel", "Power BI", "Tableau", "Looker",
    "AWS", "Azure", "GCP", "Docker", "Kubernetes", "Machine Learning",
    "Deep Learning", "TensorFlow", "PyTorch", "NLP", "Spark", "Hadoop",
    "Airflow", "ETL", "Git", "Linux", "REST API", "PostgreSQL", "MySQL",
    "MongoDB", "Snowflake", "Django", "Flask", "React", "Node.js", "Agile", "Scrum",
]

def extract_skills(description, skills_list):
    if not description:
        return []
    text = str(description).lower()
    return [s for s in skills_list if re.search(r'\b' + re.escape(s.lower()) + r'\b', text)]

df_clean["skills"] = df_clean["description"].apply(lambda d: extract_skills(d, skills_list))

In [9]:
##Insights — Top Skills & Salary
all_skills = [s for skills in df_clean["skills"] for s in skills]
top_skills = pd.DataFrame(Counter(all_skills).most_common(15), columns=["skill", "count"])
print(top_skills)

               skill  count
0   Machine Learning    128
1             Python     67
2                AWS     52
3              Azure     45
4                SQL     33
5               Java     27
6           Power BI     23
7              Agile     21
8                ETL     13
9              Excel     12
10             React      8
11             Linux      7
12               NLP      6
13             Spark      6
14           Airflow      6


In [10]:
skill_salary = {}
for skill in skills_list:
    mask = df_clean["skills"].apply(lambda s: skill in s)
    salaries = df_clean[mask]["salary_min"].dropna()
    if len(salaries) >= 5:
        skill_salary[skill] = {"avg_salary": salaries.mean(), "job_count": len(salaries)}

salary_df = pd.DataFrame(skill_salary).T.sort_values("avg_salary", ascending=False)
print(salary_df)

                     avg_salary  job_count
Kubernetes        103224.870000        6.0
Spark              96166.666667        6.0
Airflow            93500.000000        6.0
React              85875.000000        8.0
Snowflake          84614.148333        6.0
GCP                84085.286667        6.0
AWS                74867.128077       52.0
Agile              69851.935714       21.0
Python             69619.784627       67.0
Java               69436.440741       27.0
TensorFlow         69000.000000        5.0
Azure              68945.844889       45.0
PyTorch            67131.216667        6.0
ETL                67114.923846       13.0
Machine Learning   66096.861875      128.0
Deep Learning      64242.622000        5.0
SQL                56355.336667       33.0
NLP                53333.333333        6.0
Power BI           50457.040435       23.0
Linux              50398.664286        7.0
Excel              45219.173333       12.0


In [11]:
##Machine Learning Model
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df_ml = df_clean.dropna(subset=["description"])
X, y = df_ml["description"], df_ml["search_keyword"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

Accuracy: 82.29%
                               precision    recall  f1-score   support

                  AI engineer       0.79      0.73      0.76        26
             business analyst       0.80      1.00      0.89        28
business intelligence analyst       0.82      0.67      0.74        21
               cloud engineer       1.00      0.86      0.92        28
       cybersecurity engineer       0.81      0.79      0.80        28
                 data analyst       0.83      0.89      0.86        28
                data engineer       0.88      0.75      0.81        28
               data scientist       0.69      0.64      0.67        28
    machine learning engineer       0.81      0.93      0.87        28
            software engineer       0.81      0.93      0.87        28

                     accuracy                           0.82       271
                    macro avg       0.83      0.82      0.82       271
                 weighted avg       0.83      0.82      0.

In [12]:
##Export to SQL Database
companies_df = pd.DataFrame({"company_name": df_clean["company"].dropna().unique()})
companies_df["company_id"] = range(1, len(companies_df) + 1)
companies_df = companies_df[["company_id", "company_name"]]

skills_df = pd.DataFrame({"skill_name": skills_list})
skills_df["skill_id"] = range(1, len(skills_df) + 1)
skills_df = skills_df[["skill_id", "skill_name"]]

jobs_final = df_clean.merge(companies_df, left_on="company", right_on="company_name", how="left")
jobs_final = jobs_final[["job_id", "title", "company_id", "location", "salary_min",
                          "salary_max", "contract_type", "category", "description",
                          "search_keyword", "created"]]

job_skills_rows = []
for _, row in df_clean.iterrows():
    for skill in row["skills"]:
        skill_id = skills_df.loc[skills_df["skill_name"] == skill, "skill_id"].values[0]
        job_skills_rows.append({"job_id": row["job_id"], "skill_id": skill_id})
job_skills_df = pd.DataFrame(job_skills_rows)

engine_jobmarket = sa.create_engine(
    f"mssql+pyodbc://{SERVER_NAME}/JobMarketDB?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes",
    fast_executemany=True
)

companies_df.to_sql("companies", con=engine_jobmarket, if_exists="replace", index=False)
skills_df.to_sql("skills", con=engine_jobmarket, if_exists="replace", index=False)
jobs_final.to_sql("jobs_normalized", con=engine_jobmarket, if_exists="replace", index=False)
job_skills_df.to_sql("job_skills", con=engine_jobmarket, if_exists="replace", index=False)

print("Database updated!")

c:\Users\win\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Database updated!


In [13]:
export_df = df_clean.copy()
export_df["skills"] = export_df["skills"].apply(lambda x: ", ".join(x))
export_df.to_csv("../data/processed/jobs_for_dashboard.csv", index=False, encoding="utf-8-sig")
print("Dashboard CSV exported:", export_df.shape[0], "rows")

Dashboard CSV exported: 1353 rows


In [18]:
def match_jobs(user_skills, df, top_n=5):
    user_skills_set = set(s.strip().lower() for s in user_skills)
    
    def score_job(job_skills):
        job_skills_set = set(s.lower() for s in job_skills)
        if not job_skills_set:
            return 0
        overlap = user_skills_set & job_skills_set
        # نجمع بين تغطية متطلبات الوظيفة + عدد المهارات المتطابقة فعليًا
        coverage = len(overlap) / len(job_skills_set)
        matched_count = len(overlap)
        return coverage * 0.5 + (matched_count / len(user_skills_set)) * 0.5
    
    scored = df.copy()
    scored["match_score"] = scored["skills"].apply(score_job)
    scored["matched_skills"] = scored["skills"].apply(
        lambda js: [s for s in js if s.lower() in user_skills_set]
    )
    
    results = scored[scored["match_score"] > 0].sort_values("match_score", ascending=False)
    
    return results[["title", "company", "location", "salary_min", "salary_max",
                     "match_score", "matched_skills", "skills"]].head(top_n)

In [19]:
my_skills = ["Python", "SQL", "Power BI", "Excel"]
matches = match_jobs(my_skills, df_clean, top_n=10)
matches

,title,company,location,salary_min,salary_max,match_score,matched_skills,skills
760,H&H Performance Analyst,Civic Recruitment Limited,"Bermuda Park, Nuneaton",38555.74,38555.74,0.875,"[Python, SQL, Power BI]","[Python, SQL, Power BI]"
1,Data Analyst,Innoviz Limited,"Milton Keynes, Buckinghamshire",34000.00,45000.00,0.750,"[SQL, Excel, Power BI]","[SQL, Excel, Power BI, Tableau]"
126,Data Analyst,FBI &TMT,"Fareham, Hampshire",65000.00,75400.00,0.750,"[Excel, Power BI]","[Excel, Power BI]"
10,Senior Data Analyst - Marketing Ops,MCS Group,"Belfast, Northern Ireland",60000.00,60000.00,0.750,"[SQL, Power BI]","[SQL, Power BI]"
704,Business Intelligence Analyst,Solihull College & University Centre,"Bentley Heath, Solihull",39122.00,41057.00,0.750,"[SQL, Power BI]","[SQL, Power BI]"
709,Business Intelligence Analyst,Cathcart Technology,"Newbattle, Dalkeith",35000.00,40000.00,0.750,"[SQL, Power BI]","[SQL, Power BI]"
710,Business Intelligence Analyst,Unisys,UK,55617.14,55617.14,0.750,"[SQL, Power BI]","[SQL, Power BI]"
779,Senior Power BI Developer,WellData Limited,"Birmingham, West Midlands",64686.99,64686.99,0.750,"[SQL, Power BI]","[SQL, Power BI]"
133,Data Analyst,NEXT,"Leicester, Leicestershire",33000.00,NaN,0.750,"[SQL, Power BI]","[SQL, Power BI]"
780,Senior Power BI Developer,WellData Limited,"Balsall Heath, Birmingham",61471.33,61471.33,0.750,"[SQL, Power BI]","[SQL, Power BI]"


In [21]:
%whos

Variable                Type                  Data/Info
-------------------------------------------------------
APP_ID                  str                   743b995f
APP_KEY                 str                   4aa480a906f0d237202fa7d332015d4b
Counter                 type                  <class 'collections.Counter'>
LogisticRegression      type                  <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
SERVER_NAME             str                   localhost\SQLEXPRESS
TfidfVectorizer         type                  <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
X                       Series                Shape: (1353,)
X_test                  Series                Shape: (271,)
X_test_tfidf            csr_matrix            <Compressed Sparse Row sp<...>, 494)	0.1063465202138676
X_train                 Series                Shape: (1082,)
X_train_tfidf           csr_matrix            <Compressed Sparse Row sp<...>, 250)	0.1629434835410493
accuracy_score       

In [25]:
redirect_map = {j.get("id"): j.get("redirect_url") for j in all_jobs_raw}
df_clean["redirect_url"] = df_clean["job_id"].astype(str).map(redirect_map)

print(df_clean["redirect_url"].notna().sum(), "jobs have a redirect URL")

1353 jobs have a redirect URL


In [26]:
# تأكد إن العمود موجود في df_clean
print("redirect_url" in df_clean.columns)

True


In [27]:
export_df = df_clean.copy()
export_df["skills"] = export_df["skills"].apply(lambda x: ", ".join(x))

export_df.to_csv("../data/processed/jobs_for_dashboard.csv", index=False, encoding="utf-8-sig")
print("Updated with redirect_url:", export_df.shape[0], "rows")

Updated with redirect_url: 1353 rows
